# LeetCode #105: Construct Binary Tree from Preorder and Inorder Traversal

https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/

## Comparison of Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Recursive + Linear Search | O(n²) | O(n) | Linear scan to find root in inorder |
| **Recursion + HashMap ★** | **O(n)** | **O(n)** | Pre-map inorder positions for O(1) lookup |

---

## Understanding the Methods

### Brute Force (Linear Search)
At each recursive call, scan inorder array to find root position. Total O(n) calls × O(n) scan = O(n²).

### Recursion + HashMap (Optimal ★)
Pre-build a map from value → inorder index. Then `preorder[preStart]` is always the current root. Find it O(1) in the map to compute `leftSize = inRoot - inStart`. Recurse left and right with adjusted index bounds.

## Constraints
- 1 ≤ n ≤ 3000
- preorder and inorder have same elements, no duplicates

## Solutions

### C#

In [ ]:
public class Solution {
    private Dictionary<int,int> inorderMap;
    private int[] preorder;

    public TreeNode BuildTree(int[] preorder, int[] inorder) {
        this.preorder   = preorder;
        this.inorderMap = new Dictionary<int,int>();
        for (int i = 0; i < inorder.Length; i++)
            inorderMap[inorder[i]] = i;
        return Build(0, preorder.Length - 1, 0, inorder.Length - 1);
    }

    private TreeNode Build(int preStart, int preEnd, int inStart, int inEnd) {
        if (preStart > preEnd) return null;
        int rootVal  = preorder[preStart];
        int inRoot   = inorderMap[rootVal];
        int leftSize = inRoot - inStart;
        var node     = new TreeNode(rootVal);
        node.left  = Build(preStart + 1,            preStart + leftSize, inStart,    inRoot - 1);
        node.right = Build(preStart + leftSize + 1, preEnd,              inRoot + 1, inEnd);
        return node;
    }
}

### Python

In [ ]:
class Solution:
    def buildTree(self, preorder: list[int], inorder: list[int]):
        inorder_map = {v: i for i, v in enumerate(inorder)}

        def build(pre_start, pre_end, in_start, in_end):
            if pre_start > pre_end:
                return None
            root_val   = preorder[pre_start]
            in_root    = inorder_map[root_val]
            left_size  = in_root - in_start
            node       = TreeNode(root_val)
            node.left  = build(pre_start + 1,             pre_start + left_size, in_start,   in_root - 1)
            node.right = build(pre_start + left_size + 1, pre_end,               in_root + 1, in_end)
            return node

        return build(0, len(preorder) - 1, 0, len(inorder) - 1)

### Go

In [ ]:
func buildTree(preorder []int, inorder []int) *TreeNode {
    inMap := make(map[int]int, len(inorder))
    for i, v := range inorder { inMap[v] = i }

    var build func(ps, pe, is, ie int) *TreeNode
    build = func(ps, pe, is, ie int) *TreeNode {
        if ps > pe { return nil }
        rootVal  := preorder[ps]
        inRoot   := inMap[rootVal]
        leftSize := inRoot - is
        node     := &TreeNode{Val: rootVal}
        node.Left  = build(ps+1,            ps+leftSize, is,      inRoot-1)
        node.Right = build(ps+leftSize+1,   pe,          inRoot+1, ie)
        return node
    }
    return build(0, len(preorder)-1, 0, len(inorder)-1)
}

### Rust

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;
use std::collections::HashMap;

impl Solution {
    pub fn build_tree(preorder: Vec<i32>, inorder: Vec<i32>) -> Option<Rc<RefCell<TreeNode>>> {
        let in_map: HashMap<i32,usize> = inorder.iter().enumerate().map(|(i,&v)|(v,i)).collect();

        fn build(
            preorder: &[i32], ps: usize, pe: usize,
            in_map:   &HashMap<i32,usize>, is: usize, ie: usize,
        ) -> Option<Rc<RefCell<TreeNode>>> {
            if ps > pe { return None; }
            let root_val  = preorder[ps];
            let in_root   = in_map[&root_val];
            let left_size = in_root - is;
            let node = Rc::new(RefCell::new(TreeNode::new(root_val)));
            node.borrow_mut().left  = build(preorder, ps+1,            ps+left_size, in_map, is,       in_root-1);
            node.borrow_mut().right = build(preorder, ps+left_size+1,  pe,           in_map, in_root+1, ie);
            Some(node)
        }
        let n = preorder.len();
        if n == 0 { return None; }
        build(&preorder, 0, n-1, &in_map, 0, n-1)
    }
}

## Example Scenarios

### 1. Common Case — Standard Tree
**Input:** `preorder=[3,9,20,15,7], inorder=[9,3,15,20,7]`
Root=3, inRoot=1, leftSize=1. Left subtree: pre[1..1], in[0..0]. Right: pre[2..4], in[2..4].
**Output:** `[3,9,20,null,null,15,7]`

### 2. Slightly Complex — All Left Children
**Input:** `preorder=[1,2,3], inorder=[3,2,1]`
Root=1 at inorder[2]. leftSize=2. Left subtree builds 2 then 3 all to left.
**Output:** `[1,2,null,3]`

### 3. Edge Case: Time Factor — Single Node
**Input:** `preorder=[5], inorder=[5]`
preStart==preEnd, build leaf immediately.
**Output:** `[5]`

### 4. Edge Case: Space Factor — 3000-Node Perfectly Unbalanced Tree
**Input:** 3000-node right-skewed tree.
HashMap uses O(n), recursion stack O(n). Lookups all O(1).
**Output:** Correct right-skewed tree

### 5. Almost-Impossible but Plausible — Two Nodes
**Input:** `preorder=[1,2], inorder=[1,2]`
Root=1 at inorder[0]. leftSize=0. Right child=2.
**Output:** `[1,null,2]`

![image.png](attachment:image.png)